# Notebook 07: Temporal Signal Analysis
## Blink Dynamics, Yawn Duration, and Rolling PERCLOS

### Phase 2B — Physiological Signals & Temporal Analysis
This notebook investigates time-domain physiological modeling for driver fatigue assessment.

**Key Objectives:**
1. Scientific foundations of driver fatigue assessment (PERCLOS P80 criteria).
2. Time-aware processing with precise inter-frame intervals ($dt$).
3. Blink detection state machine (distinguishing normal blinks, long blinks, and micro-sleeps).
4. Sustained yawn duration filtering (isolating fatigue yawns from speech/laughter).
5. Rolling PERCLOS implementation across sliding time windows.
6. Temporal timeline visualization and comparative analysis against Phase 2A frame counters.



In [ ]:
# 1. Imports and Setup
import sys
from pathlib import Path
import csv
import matplotlib.pyplot as plt
import numpy as np

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import AppConfig
from src.physiological import BlinkAnalyzer, YawnDurationAnalyzer, PERCLOSAnalyzer

print("Temporal analysis modules loaded.")



### 2. Loading Continuous Stream Telemetry
We ingest the pre-computed frame-by-frame telemetry from `test_video.mp4` to inspect time-domain dynamics.



In [ ]:
# Ingest Telemetry Data
telemetry_path = PROJECT_ROOT / "results" / "phase2b" / "telemetry.csv"
records = []
with open(telemetry_path, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for r in reader:
        records.append(r)

timestamps = np.array([float(r["timestamp"]) for r in records])
ears = np.array([float(r["average_ear"]) if r["average_ear"] else np.nan for r in records])
mars = np.array([float(r["mar"]) if r["mar"] else np.nan for r in records])
eye_closed = np.array([r["eye_closed"].lower() == "true" for r in records])
perclos = np.array([float(r["perclos"]) * 100.0 for r in records])

print(f"Total Stream Duration: {timestamps[-1]:.2f} seconds ({len(records)} frames)")



### 3 & 4. Blink Dynamics & Micro-Sleep Classification
Normal human blinks last between $100$ ms and $400$ ms.
- **Normal Blink**: $80\text{ ms} \le t < 400\text{ ms}$.
- **Long Blink (Drowsiness Onset)**: $400\text{ ms} \le t < 800\text{ ms}$.
- **Prolonged Eye Closure (Micro-Sleep)**: $t \ge 800\text{ ms}$.



In [ ]:
# Simulate Blink State Machine with Configurable Parameters
blink_analyzer = BlinkAnalyzer(
    normal_blink_min_s=0.08,
    normal_blink_max_s=0.40,
    long_blink_max_s=0.80,
    rolling_window_s=60.0
)

events = []
for t, closed in zip(timestamps, eye_closed):
    ev = blink_analyzer.update(is_closed=bool(closed), timestamp=t)
    if ev is not None:
        events.append(ev)

print(f"Total Completed Blinks Detected: {blink_analyzer.total_blink_count}")
print(f"Total Categorized Blink Events: {len(events)}")
for ev in events[:8]:
    print(f"  Event: {ev.event_type:<18} | Duration: {ev.duration*1000:.1f} ms | At: {ev.end_time:.2f}s")



### 5. Sustained Yawn Duration Analysis
Short mouth openings ($< 1.0$s) often occur during speech, laughing, or deep breaths.
Fatigue yawns are characterized by sustained mouth aperture lasting $\ge 2.0$ seconds.



In [ ]:
# Yawn Duration Tracking
yawn_analyzer = YawnDurationAnalyzer(min_duration_s=2.0, max_duration_s=8.0, cooldown_s=3.0)

yawn_events = []
for t, m in zip(timestamps, mars):
    is_open = m > 0.55 if not np.isnan(m) else False
    st = yawn_analyzer.update(mar_is_open=is_open, yolo_yawning=is_open, timestamp=t)
    if st.event:
        yawn_events.append((t, st.event, st.current_duration))

print(f"Total Fatigue Yawns Confirmed: {yawn_analyzer.total_yawns}")
for y in yawn_events:
    print(f"  Yawn Event: {y[1]} at {y[0]:.2f}s (duration: {y[2]:.2f}s)")



### 6. Rolling PERCLOS (P80 Standard)
PERCLOS measures the proportion of time the driver's eyes are $\ge 80\%$ closed over a rolling sliding window (e.g. 15s to 60s).
$$\text{PERCLOS}(t) = \frac{\int_{t - W}^{t} \mathbb{I}_{\text{closed}}(\tau)\, d\tau}{W}$$
- **Normal**: $\text{PERCLOS} < 15\%$
- **Warning / Drowsy**: $15\% \le \text{PERCLOS} < 30\%$
- **Critical / Asleep**: $\text{PERCLOS} \ge 30\%$



In [ ]:
# Visualize Multi-Signal Temporal Timeline
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

# EAR & Closure state
ax1.plot(timestamps, ears, color="#1f77b4", label="EAR", lw=1.0)
ax1.plot(timestamps, [0.32 if c else 0.12 for c in eye_closed], color="#ff7f0e", lw=1.0, alpha=0.6, label="Closed State")
ax1.axhline(0.21, color="red", linestyle="--", lw=1.0, label="EAR Thresh (0.21)")
ax1.set_ylabel("Eye State")
ax1.set_title("Temporal Dynamics: Eye Closures, Rolling PERCLOS, and Mouth Aperture")
ax1.grid(True, alpha=0.3)
ax1.legend(loc="upper right")

# Rolling PERCLOS
ax2.plot(timestamps, perclos, color="#2ca02c", lw=1.6, label="Rolling PERCLOS (%)")
ax2.axhline(15.0, color="#ff7f0e", linestyle="--", label="Warning Threshold (15%)")
ax2.axhline(30.0, color="#d62728", linestyle="--", label="Critical Threshold (30%)")
ax2.set_ylabel("PERCLOS (%)")
ax2.grid(True, alpha=0.3)
ax2.legend(loc="upper right")

# MAR
ax3.plot(timestamps, mars, color="#9467bd", lw=1.0, label="MAR")
ax3.axhline(0.55, color="red", linestyle="--", lw=1.0, label="MAR Yawn Thresh")
ax3.set_xlabel("Time (seconds)")
ax3.set_ylabel("MAR")
ax3.grid(True, alpha=0.3)
ax3.legend(loc="upper right")

plt.tight_layout()
plt.show()



### 7. Comparison: Frame-Counter Engine vs Temporal Engine
| Dimension | Phase 2A Frame Counter | Phase 2B Temporal Signals |
| :--- | :--- | :--- |
| **Time Awareness** | Relies on fixed frame indices | Uses true timestamps ($dt$) |
| **Eye Closure** | Instantaneous threshold | Distinguishes normal blinks vs micro-sleeps |
| **Yawn Filtering** | Consecutive frame counter | Requires sustained duration $\ge 2.0$s |
| **Fatigue Metric** | Instantaneous alert state | Cumulative rolling PERCLOS window |
| **False Alarms** | Vulnerable to brief frame drops | Smooth sliding window mitigates noise |



### 8. Conclusion
Temporal analysis provides stable, physiologically sound fatigue metrics that bridge raw visual detections with clinical fatigue assessment standards.

